### Ingesting Data into Bronze Layer

In [0]:
# 1. Configuration
storage_account_name = "shivam2112"
container_name = "finalproject"
catalog_name = "shivam_catalog_f1_project"
schema_name = "bronze"

# List of tables to ingest for this batch run
tables_to_ingest = ["weather", "components", "components", "historical"]

# 2. Set up Catalog and Schema in Unity Catalog
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

print(f"Using catalog '{catalog_name}' and schema '{schema_name}'.")

# 3. Loop through and ingest each table
for table_name in tables_to_ingest:
    try:
        # Construct the source path for the Parquet file in ADLS Gen2
        source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw_data/{table_name}.parquet"
        
        # Define the full name for the destination table in Unity Catalog
        destination_table = f"{catalog_name}.{schema_name}.{table_name}"
        
        # Read the Parquet file into a DataFrame
        df = spark.read.format("parquet").load(source_path)
        
        # --- Add ingestion metadata (Good Practice for Bronze Layer) ---
        # This helps track when data was loaded and from where.
        from pyspark.sql.functions import lit, current_timestamp
        
        df_with_metadata = df.withColumn("ingestion_timestamp", current_timestamp()) \
                             .withColumn("source_file", lit(source_path))

        
        df_with_metadata.write.mode("overwrite").saveAsTable(destination_table)
        
        print(f"✅ Successfully ingested '{table_name}.parquet' into table '{destination_table}'.")

    except Exception as e:
        print(f"❌ Failed to ingest '{table_name}'. Error: {e}")

Using catalog 'shivam_catalog_f1_project' and schema 'bronze'.
✅ Successfully ingested 'weather.parquet' into table 'shivam_catalog_f1_project.bronze.weather'.
✅ Successfully ingested 'components.parquet' into table 'shivam_catalog_f1_project.bronze.components'.
✅ Successfully ingested 'drivers.parquet' into table 'shivam_catalog_f1_project.bronze.drivers'.
✅ Successfully ingested 'historical.parquet' into table 'shivam_catalog_f1_project.bronze.historical'.


###  Configuration

In [0]:
from pyspark.sql.functions import col, concat_ws, to_date, when, lit, regexp_replace
from pyspark.sql.functions import col, to_date, lower
from pyspark.sql.types import IntegerType, FloatType, DateType

# 1. Configuration
catalog_name = "shivam_catalog_f1_project"
bronze_schema = "bronze"
silver_schema = "silver"

# 2. Set up Silver Schema in Unity Catalog
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
print(f"Using catalog '{catalog_name}' and created schema '{silver_schema}'.")


Using catalog 'shivam_catalog_f1_project' and created schema 'silver'.


Weather data silver layer

In [0]:
from pyspark.sql.functions import col, to_date, lower

# 1. Configuration
catalog_name = "shivam_catalog_f1_project"
bronze_schema = "bronze"
silver_schema = "silver"

# 2. Set the current catalog and schema
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{bronze_schema}`")

print(f"Reading from '{catalog_name}.{bronze_schema}.weather'")

try:
    # 3. Read the raw weather data from the bronze table
    df_weather_bronze = spark.read.table("weather")

    # 4. Apply cleaning and transformation steps
    df_weather_silver = df_weather_bronze \
        .na.drop(subset=["date", "track","temp_c","forecast"]) \
        .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
        .withColumn("weather_description", lower(col("forecast"))) \
        .withColumnRenamed("temp_c", "temperature_celsius") \
        .select("date", "track", "weather_description", "temperature_celsius")

    # 5. Write the cleaned data to the silver table
    destination_table = f"{catalog_name}.{silver_schema}.weather"
    df_weather_silver.write.mode("overwrite").saveAsTable(destination_table)

    print(f"✅ Successfully cleaned 'weather' data and saved to '{destination_table}'.")
    
    # Optional: Show a sample of the cleaned data
    print("\nSample of cleaned data:")
    df_weather_silver.show(5)

except Exception as e:
    print(f"❌ Failed to process 'weather' data. Error: {e}")

Reading from 'shivam_catalog_f1_project.bronze.weather'
✅ Successfully cleaned 'weather' data and saved to 'shivam_catalog_f1_project.silver.weather'.

Sample of cleaned data:
+----------+-----------+-------------------+-------------------+
|      date|      track|weather_description|temperature_celsius|
+----------+-----------+-------------------+-------------------+
|2025-07-30|      Monza|               rain|               26.3|
|2025-08-01|Silverstone|              sunny|               22.5|
|2025-08-02|     Monaco|              sunny|               20.0|
|2025-08-03|        Spa|               rain|               22.8|
|2025-08-04|      Monza|              sunny|               24.1|
+----------+-----------+-------------------+-------------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, lower
from pyspark.sql.types import IntegerType

# 1. Configuration
catalog_name = "shivam_catalog_f1_project"
bronze_schema = "bronze"
silver_schema = "silver"

# 2. Set the current catalog
spark.sql(f"USE CATALOG `{catalog_name}`")

print(f"Reading from '{catalog_name}.{bronze_schema}.historical'")

try:
    # 3. Read the raw historical data from the bronze table
    df_historical_bronze = spark.read.table(f"{bronze_schema}.historical")

    # 4. Apply cleaning and transformation steps
    df_lap_data_silver = df_historical_bronze \
        .filter((col("race_id") != "INVALID_IFT") & (col("car_id") != "INVALID_LC1") & (col("weather")!="INVALID_NEU") & (col("tire")!="INVALID_GUS")) \
        .na.drop(subset=["lap", "lap_time", "car_id", "race_id","pit_stop"]) \
        .withColumn("lap", col("lap").cast(IntegerType())) \
        .withColumn("pit_stop", col("pit_stop").cast(IntegerType())) \
        .withColumn("tire_compound", lower(col("tire"))) \
        .withColumn("weather_condition", lower(col("weather"))) \
        .select(
            "race_id",
            "car_id",
            "lap",
            "lap_time",
            "pit_stop",
            "tire_compound",
            "weather_condition"
        )

    # 5. Write the cleaned data to the silver table
    destination_table = f"{silver_schema}.lap_data"
    df_lap_data_silver.write.mode("overwrite").saveAsTable(destination_table)

    print(f"✅ Successfully cleaned 'historical' data and saved to '{destination_table}'.")
    
    # Optional: Show a sample of the cleaned data
    print("\nSample of cleaned lap data:")
    df_lap_data_silver.show(10)

except Exception as e:
    print(f"❌ Failed to process 'historical' data. Error: {e}")

Reading from 'shivam_catalog_f1_project.bronze.historical'
✅ Successfully cleaned 'historical' data and saved to 'silver.lap_data'.

Sample of cleaned lap data:
+-------+------+---+--------+--------+-------------+-----------------+
|race_id|car_id|lap|lap_time|pit_stop|tire_compound|weather_condition|
+-------+------+---+--------+--------+-------------+-----------------+
|   R001|   C02|  1|  92.568|       0|         hard|            sunny|
|   R001|   C02|  2|  88.574|       0|         hard|             rain|
|   R001|   C01|  3|  87.485|       0|         soft|             rain|
|   R001|   C02|  3|  82.588|       0|         hard|            sunny|
|   R001|   C01|  4|  78.671|       0|         hard|           cloudy|
|   R001|   C02|  4|  82.728|       0|       medium|             rain|
|   R001|   C02|  5|  94.944|       0|         soft|            sunny|
|   R001|   C01|  6|  76.676|       0|         hard|             rain|
|   R001|   C02|  7|  86.946|       0|         hard|      

In [0]:
from pyspark.sql.functions import col, lower
from pyspark.sql.types import IntegerType

# 1. Configuration
catalog_name = "shivam_catalog_f1_project"
bronze_schema = "bronze"
silver_schema = "silver"

# 2. Set the current catalog
spark.sql(f"USE CATALOG `{catalog_name}`")

print(f"Reading from '{catalog_name}.{bronze_schema}.components")

try:
    # 3. Read the raw historical data from the bronze table
    df_components_bronze = spark.read.table(f"{bronze_schema}.components")

    # 4. Apply cleaning and transformation steps
    df_components_silver = df_components_bronze

    # 5. Write the cleaned data to the silver table
    destination_table = f"{silver_schema}.components"
    df_lap_data_silver.write.mode("overwrite").saveAsTable(destination_table)

    print(f"✅ Successfully cleaned 'drivers' data and saved to '{destination_table}'.")
    
    # Optional: Show a sample of the cleaned data
    print("\nSample of cleaned lap data:")
    df_components_silver.show(10)

except Exception as e:
    print(f"❌ Failed to process 'historical' data. Error: {e}")

Reading from 'shivam_catalog_f1_project.bronze.drivers
✅ Successfully cleaned 'drivers' data and saved to 'silver.drivers'.

Sample of cleaned lap data:
+------+--------------+--------+-------+--------------------+--------------------+
|car_id|   driver_name|    team| engine| ingestion_timestamp|         source_file|
+------+--------------+--------+-------+--------------------+--------------------+
|   C01|Lewis Hamilton|Mercedes|Ferrari|2025-08-30 11:13:...|abfss://finalproj...|
|   C02|  Lando Norris|Mercedes|Ferrari|2025-08-30 11:13:...|abfss://finalproj...|
+------+--------------+--------+-------+--------------------+--------------------+



In [0]:
from pyspark.sql.functions import col, lower
from pyspark.sql.types import IntegerType

# 1. Configuration
catalog_name = "shivam_catalog_f1_project"
bronze_schema = "bronze"
silver_schema = "silver"

# 2. Set the current catalog
spark.sql(f"USE CATALOG `{catalog_name}`")

print(f"Reading from '{catalog_name}.{bronze_schema}.components")

try:
    # 3. Read the raw historical data from the bronze table
    df_components_bronze = spark.read.table(f"{bronze_schema}.components")

    # 4. Apply cleaning and transformation steps
    df_components_silver = df_components_bronze

    # 5. Write the cleaned data to the silver table
    destination_table = f"{silver_schema}.components"
    df_lap_data_silver.write.mode("overwrite").saveAsTable(destination_table)

    print(f"✅ Successfully cleaned 'components' data and saved to '{destination_table}'.")
    
    # Optional: Show a sample of the cleaned data
    print("\nSample of cleaned lap data:")
    df_components_silver.show(10)

except Exception as e:
    print(f"❌ Failed to process 'historical' data. Error: {e}")

Reading from 'shivam_catalog_f1_project.bronze.components
✅ Successfully cleaned 'components' data and saved to 'silver.components'.

Sample of cleaned lap data:
+------+-------+--------------+--------------+--------------------+--------------------+
|car_id|chassis|  aero_package|engine_version| ingestion_timestamp|         source_file|
+------+-------+--------------+--------------+--------------------+--------------------+
|   C01|  SpecA|High-Downforce|        EV_KNQ|2025-08-30 11:13:...|abfss://finalproj...|
|   C02|  SpecA| Low-Downforce|        EV_MIQ|2025-08-30 11:13:...|abfss://finalproj...|
+------+-------+--------------+--------------+--------------------+--------------------+



In [0]:
from pyspark.sql.functions import col, avg, min, max, sum, collect_set, lag, row_number, first, stddev
from pyspark.sql.functions import col, avg, min, max, sum, count, collect_set, lag, row_number
from pyspark.sql.window import Window

# 1. Configuration
catalog_name = "shivam_catalog_f1_project"
silver_schema = "silver"
gold_schema = "gold"

# 2. Set up Gold Schema in Unity Catalog
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

# 3. Read the silver table
lap_data_df = spark.read.table(f"{silver_schema}.lap_data")
# ==============================================================================
# Gold Table 1: Race Summary for BI and Analytics
# ==============================================================================
print("\n--- Creating Gold Table: race_summary ---")
try:
    race_summary_df = lap_data_df \
        .groupBy("race_id", "car_id") \
        .agg(
            count("lap").alias("total_laps"),
            min("lap_time").alias("fastest_lap_time"),
            avg("lap_time").alias("average_lap_time"),
            sum("pit_stop").alias("total_pit_stops"),
            collect_set("tire_compound").alias("tire_compounds_used")
        )

    # Write to Gold layer
    destination_table = f"{gold_schema}.race_summary"
    race_summary_df.write.mode("overwrite").saveAsTable(destination_table)
    print(f"✅ Successfully created aggregated table '{destination_table}'.")
    race_summary_df.show(5, truncate=False)

except Exception as e:
    print(f"❌ Failed to create 'race_summary' table. Error: {e}")

# ==============================================================================
# Enhanced Gold Table: Feature Engineering for Pit Stop Prediction (ML)
# ==============================================================================

print("\n--- Creating Enhanced Gold Table: pit_stop_features ---")
try:
    # --- Part 1: Initial feature calculation (stint_id, tire_age) ---
    window_spec_by_lap = Window.partitionBy("race_id", "car_id").orderBy("lap")
    features_df_part1 = lap_data_df \
        .withColumn("stint_id", sum("pit_stop").over(window_spec_by_lap) + 1)
    window_spec_by_stint = Window.partitionBy("race_id", "car_id", "stint_id").orderBy("lap")
    features_df_part1 = features_df_part1 \
        .withColumn("tire_age_laps", row_number().over(window_spec_by_stint))
    # --- Part 2: Calculate baseline and average metrics ---
    # Calculate the baseline pace (avg of first 2 laps) for each stint
    stint_baseline_df = features_df_part1 \
        .filter(col("tire_age_laps") <= 2) \
        .groupBy("race_id", "car_id", "stint_id") \
        .agg(avg("lap_time").alias("stint_baseline_lap_time"))
    # Calculate the average pace for the entire race for each car
    race_avg_df = features_df_part1 \
        .groupBy("race_id", "car_id") \
        .agg(avg("lap_time").alias("race_avg_lap_time"))
    # --- Part 3: Join metrics back and calculate final features ---
    final_features_df = features_df_part1 \
        .join(stint_baseline_df, ["race_id", "car_id", "stint_id"], "left") \
        .join(race_avg_df, ["race_id", "car_id"], "left") \
        .orderBy("race_id", "car_id", "lap") \
        .withColumn("lap_time_degradation", col("lap_time") - col("stint_baseline_lap_time")) \
        .withColumn("lap_time_vs_race_avg", col("lap_time") - col("race_avg_lap_time")) \
        .withColumn("lap_time_volatility", stddev("lap_time").over(window_spec_by_lap.rowsBetween(-3, 0))) # 4-lap rolling std dev
    # --- Part 4: Final Selection of Columns for the model ---
    model_input_df = final_features_df.select(
        "race_id",
        "car_id",
        "lap",
        "lap_time",
        "stint_id",
        "tire_age_laps",
        "stint_baseline_lap_time",
        "lap_time_degradation",
        "lap_time_vs_race_avg",
        "lap_time_volatility",
        "tire_compound",
        "weather_condition",
        "pit_stop"  # This is the label for the ML model
    ).na.fill(0) # Fill any initial nulls (e.g., for volatility on lap 1) with 0

    # Write to Gold layer
    destination_table = f"{gold_schema}.pit_stop_features"
    model_input_df.write.mode("overwrite").saveAsTable(destination_table)
    print(f"✅ Successfully created enhanced feature table '{destination_table}'.")
    model_input_df.show(10)

except Exception as e:
    print(f"❌ Failed to create 'pit_stop_features' table. Error: {e}")


--- Creating Gold Table: race_summary ---
✅ Successfully created aggregated table 'gold.race_summary'.
+-------+------+----------+----------------+------------------+---------------+--------------------+
|race_id|car_id|total_laps|fastest_lap_time|average_lap_time  |total_pit_stops|tire_compounds_used |
+-------+------+----------+----------------+------------------+---------------+--------------------+
|R005   |C02   |45        |75.593          |120.68911340996172|1              |[hard, medium, soft]|
|R004   |C01   |43        |75.251          |87.29693023255817 |0              |[hard, soft, medium]|
|R004   |C02   |42        |75.691          |84.6977857142857  |0              |[hard, medium, soft]|
|R005   |C01   |38        |78.709          |128.78852903811253|0              |[hard, soft, medium]|
|R002   |C01   |46        |77.454          |120.29858920539733|0              |[hard, soft, medium]|
+-------+------+----------+----------------+------------------+---------------+---------